## Exploración masiva de Mapper para el dataset de ácido fólico en mujeres chilenas embarazadas.
 
Genera TODAS las combinaciones de:
  - Lentes (filtros de proyección)
  - Cover (n_cubes × perc_overlap)
  - Clustering DBSCAN (eps × min_samples)
  - Espacio métrico (continuas std, continuas robust, todas std)

In [4]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import os, itertools, time
from pathlib import Path
 
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
import umap
 
import kmapper as km
from kmapper import Cover
import networkx as nx

## 1. Configuración

In [2]:
DATASET_PATH = Path("data") / "dataset2_limpio_ohe.csv"
OUTPUT_DIR   = Path("resultados")
TOP_DIR      = OUTPUT_DIR / "top_mappers"
BIO_DIR      = OUTPUT_DIR / "top_bio"

for d in [OUTPUT_DIR, TOP_DIR, BIO_DIR]:
    d.mkdir(exist_ok=True, parents=True)

TOP_N = 20

## 2. Datos

In [5]:
df = pd.read_csv(DATASET_PATH).drop(
    columns=["Unnamed: 0","n","Fecha encuesta","FN hijo", "Fecha Encuesta"], errors="ignore"
)

# Truco: Quitamos espacios dobles raros de los nombres de las columnas del dataframe
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Lista de columnas continuas oficiales (sin espacios dobles extraños en el texto)
continuous_cols_raw = [
    "Edad madre","N° embarazo","PN hijo (g)","EG hijo (sem)",
    "KG inicio mamá","KG fin mamá","Dif peso mamá","Estatura mamá",
    "IMC antes","IMC después","Dif IMC",
    "mgAF/cáp SAF+OSAF","¿Cuántos días de SAF?","mgAF/día SAF",
    "mgAF/cáp MAF","¿Cuántos días de MAF?","mgAF/día MAF 1°T",
    "mgAF/cáp OSAF","¿Cuántos días de OSAF?","mgAF/día OSAF 1°T",
    "mgAF/día SAF+OSAF 2°T",
    "mgAF/día Suplementos y multivitamínico 1°T",
    "mg/día DFE suplementos y multivitamínico 1°T",
    "Días consumo MAF","mgAF/día período MAF",
    "Días consumo suplementosAF","mgAF/día total período SAF+OSAF",
    "TOTAL AF mg suplementos y multivitamínicos período",
    "mgAF/unidad pan","mg/d AF total pan","mg/d DFE total pan",
    "Total mg/d AF suple y pan","Total mg/d DFE suple y pan"
]
# Limpiamos también nuestra lista manual para asegurar match perfecto
continuous_cols_raw = [pd.Series(c).str.replace(r'\s+', ' ', regex=True).str.strip()[0] for c in continuous_cols_raw]

# Filtrar solo las que realmente existan en el CSV
continuous_cols = [c for c in continuous_cols_raw if c in df.columns]

# Identificar cuáles faltaron por si necesitas revisar ortografía
missing_cols = [c for c in continuous_cols_raw if c not in df.columns]
if missing_cols:
    print(f"⚠️ Nota: {len(missing_cols)} columnas no se encontraron en el CSV (ej: {missing_cols[:2]})")

# Generar matrices numéricas limpias
X_cont = df[continuous_cols].fillna(0).values
X_all  = df.fillna(0).values

In [6]:
# Para las matrices que son PURAMENTE continuas (X_cont), 
# StandardScaler y RobustScaler son perfectos.
std_c = StandardScaler()
rob_c = RobustScaler()
X_cont_std = std_c.fit_transform(X_cont)
X_cont_rob = rob_c.fit_transform(X_cont)

# Para la matriz global (X_all) que contiene continuas + One-Hot Encoding:
# ¡USAMOS MINMAXSCALER! Así todo queda estrictamente entre 0 y 1 y no se distorsionan las categóricas.
mms_a = MinMaxScaler()
X_all_scaled = mms_a.fit_transform(X_all)

print(f"\n¡Listo y corregido!")
print(f" Filas: {len(df)} | Continuas detectadas: {len(continuous_cols)}")
print(f" Rango de la matriz global escalada: Mín={X_all_scaled.min()} | Máx={X_all_scaled.max()}")


¡Listo y corregido!
 Filas: 705 | Continuas detectadas: 33
 Rango de la matriz global escalada: Mín=0.0 | Máx=1.0000000000000004


## 3. Declaración de Opciones

### 3.1 Variables de Coloreo

In [7]:
color_vars = {
    "mgAF_dia_total"  : df["Total mg/d AF suple y pan"].fillna(0).values,
    "mgAF_suple_1T"   : df["mgAF/día Suplementos y multivitamínico 1°T"].fillna(0).values,
    "mgAF_saf"        : df["mgAF/día SAF"].fillna(0).values,
    "total_AF_mg"     : df["TOTAL AF mg suplementos y multivitamínicos período"].fillna(0).values,
    "dias_saf"        : df["Días consumo suplementosAF"].fillna(0).values,
    "pn_hijo"         : df["PN hijo (g)"].fillna(0).values,
    "eg_hijo"         : df["EG hijo (sem)"].fillna(0).values,
    "edad_madre"      : df["Edad madre"].fillna(0).values,
    "imc_antes"       : df["IMC antes"].fillna(0).values,
    "dif_imc"         : df["Dif IMC"].fillna(0).values,
    
    # Forzamos las binarias a enteros (0 y 1) para que KeplerMapper pueda sacar promedios visuales limpios
    "hijo_problema"   : df["¿Hijo nace c/problema de salud?"].astype(int).fillna(0).values,
    "consume_suple"   : df["¿Consume suplementos y/o multivitamínico de AF en embarazo?"].astype(int).fillna(0).values,
}

### 3.2 Lentes

In [8]:
# Función auxiliar de escalamiento seguro
def ss(X): 
    return StandardScaler().fit_transform(X)

# PCA puramente continuo
pca2 = PCA(n_components=2).fit_transform(X_cont_std)

# PCA mixto: ¡Usamos X_all_scaled (MinMaxScaler) para no sesgar con las OHE!
pca_full2d = PCA(n_components=2).fit_transform(X_all_scaled)

u_15_02 = umap.UMAP(n_components=1, n_neighbors=15, min_dist=0.2, random_state=42).fit_transform(X_cont_std)
u_30_01 = umap.UMAP(n_components=1, n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(X_cont_std)
u_5_05  = umap.UMAP(n_components=1, n_neighbors=5,  min_dist=0.5, random_state=42).fit_transform(X_cont_std)
u_2d    = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_cont_std)

# Densidad local (mide qué tan aisladas están las pacientes)
nbrs = NearestNeighbors(n_neighbors=10).fit(X_cont_std)
dists, _ = nbrs.kneighbors(X_cont_std)
lens_density = ss(-dists.mean(axis=1, keepdims=True))

# Excentricidad (mide qué tan lejos está cada paciente del centro geométrico)
D = pairwise_distances(X_cont_std, metric="euclidean")
lens_eccentric = ss(D.mean(axis=1, keepdims=True))

# Lentes biológicos individuales escalados (con nombres corregidos sin espacios raros)
lens_af_total = ss(df[["Total mg/d AF suple y pan"]].fillna(0).values)

# Lente Híbrido (UMAP 1D + Ácido Fólico Total)
# Une la geometría global de los datos con el consumo nutricional en un lente 2D
lens_hibrido_umap_af = np.hstack([u_15_02, lens_af_total])

lenses = {
    "pca_1"          : pca2[:, :1],
    "pca_2"          : pca2[:, 1:2],
    "pca_2d"         : pca2,
    "pca_full_2d"    : pca_full2d,
    "umap_15_02"     : u_15_02,
    "umap_30_01"     : u_30_01,
    "umap_5_05"      : u_5_05,
    "umap_2d"        : u_2d,
    "hibrido_umap_af": lens_hibrido_umap_af,
    
    # Nombres de columnas corregidos y protegidos con fillna(0)
    "mgAF_total"     : lens_af_total,
    "mgAF_suple_1T"  : ss(df[["mgAF/día Suplementos y multivitamínico 1°T"]].fillna(0).values),
    "mgAF_saf"       : ss(df[["mgAF/día SAF"]].fillna(0).values),
    "total_AF_mg"    : ss(df[["TOTAL AF mg suplementos y multivitamínicos período"]].fillna(0).values),
    "imc_antes"      : ss(df[["IMC antes"]].fillna(0).values),
    "eg_hijo"        : ss(df[["EG hijo (sem)"]].fillna(0).values),
    "pn_hijo"        : ss(df[["PN hijo (g)"]].fillna(0).values),
    "edad_madre"     : ss(df[["Edad madre"]].fillna(0).values),
    "dif_imc"        : ss(df[["Dif IMC"]].fillna(0).values),
    "dias_saf"       : ss(df[["Días consumo suplementosAF"]].fillna(0).values),
    "densidad"       : lens_density,
    "excentricidad"  : lens_eccentric,
}

print(f"  ¡Listo! Total lentes configurados con éxito: {len(lenses)}")

  ¡Listo! Total lentes configurados con éxito: 21


### 3.3 Grilla de Hiperparámetros

In [9]:
# Configuración de los Cubos (Resolución del grafo)
n_cubes_list     = [8, 12, 16, 20]

# Porcentaje de traslape (Conectividad del grafo)
overlap_list     = [0.2, 0.3, 0.4, 0.5, 0.6]

# Radio de DBSCAN ajustado a la geometría real de tus 176 dimensiones
eps_list         = [1.0, 1.5, 2.0, 2.5, 3.5, 5.0]

# Cantidad mínima de mujeres para formar una subpoblación/nodo
min_samples_list = [2, 3, 5]

# Espacios métricos actualizados con la corrección de MinMaxScaler
metric_spaces = {
    "cont_std"  : X_cont_std,
    "cont_rob"  : X_cont_rob,
    "all_mixed" : X_all_scaled
}

# Cálculo del nuevo total de combinaciones optimizadas
total = (len(lenses) * len(metric_spaces) *
         len(n_cubes_list) * len(overlap_list) *
         len(eps_list) * len(min_samples_list))

print(f"Total de combinaciones optimizadas a procesar: {total:,}")

Total de combinaciones optimizadas a procesar: 22,680


## 4. Métricas

In [10]:
def compute_metrics(graph, n_pts):
    nodes = graph["nodes"]
    links = graph["links"]
    n_nodes = len(nodes)
    if n_nodes < 2:
        return None
 
    G = nx.Graph()
    G.add_nodes_from(nodes.keys())
    for src, tgts in links.items():
        for tgt in tgts:
            G.add_edge(src, tgt)
 
    n_edges = G.number_of_edges()
    n_comp  = nx.number_connected_components(G)
    n_loops = max(0, n_edges - n_nodes + n_comp)
    degs    = [d for _, d in G.degree()]
    sizes   = np.array([len(v) for v in nodes.values()])
    probs   = sizes / sizes.sum()
    entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
    collapse = sizes.max() / n_pts
 
    score = (
        0.25 * min(n_nodes / 40, 1.0)
      + 0.25 * min(n_loops / 3, 1.0)
      + 0.20 * min(n_comp / 5, 1.0)
      + 0.20 * min(entropy / 3.5, 1.0)
      - 0.10 * collapse
    )
    return dict(
        n_nodes=n_nodes, n_edges=n_edges, n_components=n_comp,
        n_loops=n_loops,
        avg_degree=round(float(np.mean(degs)),2),
        max_degree=int(np.max(degs)),
        density=round(float(nx.density(G)),4),
        avg_node_size=round(float(sizes.mean()),1),
        max_node_size=int(sizes.max()),
        size_entropy=round(entropy,3),
        collapse_ratio=round(float(collapse),3),
        score=round(score,4),
    )
 
def inter_node_variance(graph, arr):
    if len(graph["nodes"]) <= 1:
        return 0.0
    
    means = []
    for idx in graph["nodes"].values():
        if len(idx) > 0:
            valores_nodo = arr[idx]
            means.append(valores_nodo.mean())
            
    return round(float(np.var(means)), 4) if len(means) > 1 else 0.0

## 5. Grid Search Mappers al Fallo

### 5.1 Grid Search

In [11]:
mapper = km.KeplerMapper(verbose=0)
results = []
t0 = time.time()
idx = 0

for (lens_name, lens_data), (space_name, X_space) in itertools.product(
        lenses.items(), metric_spaces.items()):

    for n_cubes, overlap in itertools.product(n_cubes_list, overlap_list):
        cover = Cover(n_cubes=n_cubes, perc_overlap=overlap)

        for eps, min_samp in itertools.product(eps_list, min_samples_list):
            try:
                # Generar el grafo con la combinación actual
                graph = mapper.map(
                    lens_data,
                    X_space,
                    cover=cover,
                    clusterer=DBSCAN(eps=eps, min_samples=min_samp),
                )
                
                # Calcular métricas estructurales
                m = compute_metrics(graph, len(df))
                
                # Si el grafo es válido, calculamos las métricas clínicas
                if m is not None:
                    bio_vars = {}
                    for cname, cvals in color_vars.items():
                        bio_vars[f"var_{cname}"] = inter_node_variance(graph, cvals)

                    # Guarda el diccionario completo
                    results.append(dict(
                        id=idx,
                        lens=lens_name, metric_space=space_name,
                        n_cubes=n_cubes, overlap=overlap,
                        eps=eps, min_samples=min_samp,
                        **m, **bio_vars,
                    ))
            except Exception:
                # Si un clusterer truaca por geometría extrema, lo ignoramos de forma segura
                pass
            
            idx += 1
            if idx % 1000 == 0:
                el = time.time() - t0
                eta = (el / idx) * (total - idx)
                print(f"  [{idx:,}/{total:,}] {idx/total*100:.1f}%  "
                      f"elapsed={el/60:.1f}min  ETA={eta/60:.1f}min  "
                      f"válidos={len(results):,}")

print(f"\n>> Búsqueda finalizada: {len(results):,} grafos válidos de {total:,} intentos")
print(f"Tiempo total de cómputo: {(time.time()-t0)/60:.1f} min")

  [1,000/22,680] 4.4%  elapsed=1.6min  ETA=33.6min  válidos=862
  [2,000/22,680] 8.8%  elapsed=2.9min  ETA=29.7min  válidos=1,701
  [3,000/22,680] 13.2%  elapsed=13.9min  ETA=91.0min  válidos=2,542
  [4,000/22,680] 17.6%  elapsed=18.2min  ETA=85.0min  válidos=3,388
  [5,000/22,680] 22.0%  elapsed=20.5min  ETA=72.5min  válidos=4,257
  [6,000/22,680] 26.5%  elapsed=21.2min  ETA=58.9min  válidos=5,121
  [7,000/22,680] 30.9%  elapsed=22.0min  ETA=49.3min  válidos=5,980
  [8,000/22,680] 35.3%  elapsed=24.2min  ETA=44.3min  válidos=6,836
  [9,000/22,680] 39.7%  elapsed=27.9min  ETA=42.4min  válidos=7,687
  [10,000/22,680] 44.1%  elapsed=30.3min  ETA=38.5min  válidos=8,557
  [11,000/22,680] 48.5%  elapsed=31.2min  ETA=33.1min  válidos=9,428
  [12,000/22,680] 52.9%  elapsed=32.1min  ETA=28.5min  válidos=10,294
  [13,000/22,680] 57.3%  elapsed=32.7min  ETA=24.3min  válidos=11,164
  [14,000/22,680] 61.7%  elapsed=33.4min  ETA=20.7min  válidos=12,032
  [15,000/22,680] 66.1%  elapsed=34.0min  ETA=

In [12]:
if results:
    # Convertimos a DataFrame, ordenamos por score topológico de mayor a menor
    df_res = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
    
    csv_path = OUTPUT_DIR / "resultados_mapper.csv"
    df_res.to_csv(csv_path, index=False)
    print(f"\n✅ CSV guardado exitosamente en: {csv_path} ({len(df_res):,} filas)")

    # Columnas clave para la vista en consola
    show_cols = ["lens", "metric_space", "n_cubes", "overlap", "eps", "min_samples",
                 "n_nodes", "n_edges", "n_components", "n_loops", "score"]
    
    print("\n🏆 ── TOP 15 GRAFOS POR SCORE STRUCTURAL ──")
    print(df_res[show_cols].head(15).to_string(index=False))
else:
    print("\n❌ ¡Alerta crítica! No se generó ningún grafo válido en las 22,680 combinaciones.")
    print("Por favor, revisa que los datos de entrada en metric_spaces no contengan valores infinitos.")


✅ CSV guardado exitosamente en: resultados\resultados_mapper.csv (19,236 filas)

🏆 ── TOP 15 GRAFOS POR SCORE STRUCTURAL ──
           lens metric_space  n_cubes  overlap  eps  min_samples  n_nodes  n_edges  n_components  n_loops  score
    pca_full_2d    all_mixed       20      0.6  1.0            2       43      144             6      107 0.8997
        umap_2d    all_mixed       20      0.4  1.5            2       52       43            24       15 0.8994
     edad_madre    all_mixed       12      0.6  2.0            2       64       33            37        6 0.8993
     edad_madre    all_mixed       20      0.6  2.0            2       42       22            24        4 0.8993
     edad_madre    all_mixed       16      0.6  2.0            2       53       28            29        4 0.8993
          pca_2     cont_std       12      0.6  1.5            2       44       34            18        8 0.8991
      umap_5_05     cont_std       20      0.6  1.5            2       45       36  

### 5.2 Score Biológico

In [14]:
bio_score_cols = [
    "var_mgAF_dia_total", 
    "var_mgAF_suple_1T", 
    "var_total_AF_mg",
    "var_pn_hijo", 
    "var_eg_hijo", 
    "var_hijo_problema"
]

# Filtrar solo las columnas que realmente existan en los resultados
bio_score_cols = [c for c in bio_score_cols if c in df_res.columns]

if len(bio_score_cols) > 0:
    # Normalizar cada columna de varianza a rango [0,1] y promediar
    bio_norm = df_res[bio_score_cols].copy()
    
    for c in bio_score_cols:
        col_min = bio_norm[c].min()
        col_max = bio_norm[c].max()
        rng = col_max - col_min
        
        # Si el rango es cero (todos los grafos dieron la misma varianza), dejamos la columna en 0
        if rng == 0:
            bio_norm[c] = 0.0
        else:
            bio_norm[c] = (bio_norm[c] - col_min) / rng

    # El bio_score final es el promedio de las varianzas normalizadas
    df_res["bio_score"] = bio_norm.mean(axis=1)
    
    # Ordenar DataFrame global de mayor a menor relevancia biológica
    df_res_bio = df_res.sort_values("bio_score", ascending=False).reset_index(drop=True)
    
    # Ordenar también el df_res original por el score estructural para mantener consistencia
    df_res = df_res.sort_values("score", ascending=False).reset_index(drop=True)
    
    # Mostrar resultados en consola
    print("\n🏆 ── TOP 15 GRAFOS POR BIO_SCORE (RELEVANCIA CLÍNICA) ──")
    show_bio_cols = show_cols + ["var_mgAF_dia_total", "var_pn_hijo", "bio_score"]
    # Nos aseguramos de que solo imprima columnas que existan para que no truene la consola
    show_bio_cols = [c for c in show_bio_cols if c in df_res_bio.columns]
    print(df_res_bio[show_bio_cols].head(15).to_string(index=False))
    
    # Guardar el CSV final actualizado con ambas métricas (score y bio_score)
    df_res.to_csv(csv_path, index=False)
    print(f"\n✅ CSV actualizado y guardado con éxito en: {csv_path}")
else:
    print("\n⚠️ No se encontraron columnas de varianza biológica en df_res. Revisa el Loop Principal.")


🏆 ── TOP 15 GRAFOS POR BIO_SCORE (RELEVANCIA CLÍNICA) ──
    lens metric_space  n_cubes  overlap  eps  min_samples  n_nodes  n_edges  n_components  n_loops  score  var_mgAF_dia_total  var_pn_hijo  bio_score
 eg_hijo     cont_std        8      0.2  5.0            2       12        5             7        0 0.2367              4.8620  340937.5382   0.554485
 eg_hijo     cont_std        8      0.3  5.0            2       12        5             7        0 0.2315              4.8656  339173.0675   0.554001
 eg_hijo     cont_std       16      0.5  5.0            2       22       15             7        0 0.3362              4.6546  372311.3001   0.545719
 eg_hijo     cont_std       16      0.4  5.0            2       17       10             7        0 0.2954              5.0676  344385.6658   0.535732
 eg_hijo     cont_std       12      0.3  5.0            2       16        9             7        0 0.2895              5.1674  378683.5322   0.523375
 eg_hijo     cont_std       12      0.2  5

### 5.3 HTMLS Interactivos

In [15]:
# 1. Configuración de parámetros de exportación
TOP_N = 10

# Definimos y aseguramos las rutas hijas dentro de tu directorio de salida (OUTPUT_DIR)
TOP_DIR = OUTPUT_DIR / "top_estructura"
BIO_DIR = OUTPUT_DIR / "top_biologia"

def make_html(row, out_dir, rank, label):

    out_dir.mkdir(parents=True, exist_ok=True)
    
    lens_data = lenses[row["lens"]]
    X_space   = metric_spaces[row["metric_space"]]
    
    # Configuramos la resolución del cubo y traslape
    cover = Cover(n_cubes=int(row["n_cubes"]), perc_overlap=float(row["overlap"]))
    
    # Recomputamos el grafo específico para exportarlo
    graph = mapper.map(
        lens_data, 
        X_space, 
        cover=cover,
        clusterer=DBSCAN(eps=float(row["eps"]), min_samples=int(row["min_samples"]))
    )

    ttips = pd.DataFrame({
        "mgAF_total"    : color_vars["mgAF_dia_total"],
        "EG_sem"        : color_vars["eg_hijo"],
        "PN_g"          : color_vars["pn_hijo"],
        "edad"          : color_vars["edad_madre"],
        "IMC"           : color_vars["imc_antes"],
        "suple"         : color_vars["consume_suple"],
        "prob_salud"    : color_vars["hijo_problema"],
    }).apply(
        lambda r: (f"mgAF={r.mgAF_total:.2f}mg | "
                   f"EG={r.EG_sem:.0f}sem | PN={r.PN_g:.0f}g | "
                   f"Edad={r.edad:.0f} | IMC={r.IMC:.1f} | "
                   f"suple={'Sí' if r.suple else 'No'} | "
                   f"prob={'Sí' if r.prob_salud else 'No'}"),
        axis=1,
    ).values
 
    fname = (f"{label}_rank{rank:03d}_{row['lens']}_"
             f"{row['metric_space']}_c{int(row['n_cubes'])}_"
             f"o{row['overlap']}_eps{row['eps']}_ms{int(row['min_samples'])}.html")
    fpath = out_dir / fname
    
    # Renderizado y guardado del archivo HTML interactivo
    mapper.visualize(
        graph,
        title=(f"#{rank} | Lente={row['lens']} | Espacio={row['metric_space']} | "
               f"Cubos={int(row['n_cubes'])} | Overlap={row['overlap']} | "
               f"eps={row['eps']} | ms={int(row['min_samples'])} || "
               f"Score Estructural={row.get('score', 0):.3f} | Bio_Score={row.get('bio_score', 0):.3f}"),
        color_values=color_vars["mgAF_dia_total"],
        color_function_name="mgAF/día total (mg)",
        custom_tooltips=ttips,
        path_html=str(fpath),
    )
    return fname
 
# 2. Bucle de renderizado para el TOP por Estructura Topológica
print(f"\nGenerando {TOP_N} HTMLs basados en Score Estructural...")
for rank, row in df_res.head(TOP_N).reset_index(drop=True).iterrows():
    try:
        fname = make_html(row, TOP_DIR, rank+1, "top_struct")
        print(f"  [Estructura] {rank+1:02d}. {fname}")
    except Exception as e:
        print(f"  ❌ Error al generar HTML Estructural {rank+1}: {e}")
 
# 3. Bucle de renderizado para el TOP por Relevancia Biológica (Varianzas)
print(f"\nGenerando {TOP_N} HTMLs basados en Score de Relevancia Biológica...")
for rank, row in df_res_bio.head(TOP_N).reset_index(drop=True).iterrows():
    try:
        fname = make_html(row, BIO_DIR, rank+1, "top_bio")
        print(f"  [Biología] {rank+1:02d}. {fname}")
    except Exception as e:
        print(f"  ❌ Error al generar HTML Biológico {rank+1}: {e}")

print("\n🎉 ¡Proceso de Grid Search y exportación completado exitosamente al 100%!")
print(f"Revisa las carpetas '{TOP_DIR.name}' y '{BIO_DIR.name}' para explorar tus grafos interactivos.")


Generando 10 HTMLs basados en Score Estructural...
  [Estructura] 01. top_struct_rank001_pca_full_2d_all_mixed_c20_o0.6_eps1.0_ms2.html
  [Estructura] 02. top_struct_rank002_umap_2d_all_mixed_c20_o0.4_eps1.5_ms2.html
  [Estructura] 03. top_struct_rank003_edad_madre_all_mixed_c12_o0.6_eps2.0_ms2.html
  [Estructura] 04. top_struct_rank004_edad_madre_all_mixed_c16_o0.6_eps2.0_ms2.html
  [Estructura] 05. top_struct_rank005_edad_madre_all_mixed_c20_o0.6_eps2.0_ms2.html
  [Estructura] 06. top_struct_rank006_eg_hijo_cont_std_c16_o0.6_eps1.5_ms2.html
  [Estructura] 07. top_struct_rank007_umap_5_05_cont_std_c12_o0.6_eps1.5_ms2.html
  [Estructura] 08. top_struct_rank008_imc_antes_cont_std_c8_o0.6_eps1.5_ms2.html
  [Estructura] 09. top_struct_rank009_pca_2d_cont_std_c16_o0.6_eps1.5_ms2.html
  [Estructura] 10. top_struct_rank010_pca_2d_cont_std_c12_o0.4_eps1.5_ms2.html

Generando 10 HTMLs basados en Score de Relevancia Biológica...
  [Biología] 01. top_bio_rank001_eg_hijo_cont_std_c8_o0.2_eps5.0_

In [19]:
print("\n==============================================================")
print("RESUMEN ESTADÍSTICO DE LOS MEJORES PARÁMETROS")
print("==============================================================")

if 'df_res' in locals() and not df_res.empty:
    # ─── ANÁLISIS DEL TOP 50 POR SCORE ESTRUCTURAL ───
    print("\n TOP 50 POR CALIDAD ESTRUCTURAL (Fórmula Matemática)")
    print("--------------------------------------------------------------")
    print(" Lentes más frecuentes:")
    print(df_res.head(50)["lens"].value_counts().to_string())
    
    print("\n Resolución (n_cubes) más frecuente:")
    print(df_res.head(50)["n_cubes"].value_counts().to_string())
    
    print("\n Radio DBSCAN (eps) más frecuente:")
    print(df_res.head(50)["eps"].value_counts().to_string())

    # ─── ANÁLISIS DEL TOP 50 POR RELEVANCIA BIOLÓGICA ───
    if 'df_res_bio' in locals() and not df_res_bio.empty:
        print("\n\n TOP 50 POR RELEVANCIA BIOLÓGICA (Varianza Nutricional/Clínica)")
        print("--------------------------------------------------------------")
        print(" Lentes más frecuentes:")
        print(df_res_bio.head(50)["lens"].value_counts().to_string())
        
        print("\n Resolución (n_cubes) más frecuente:")
        print(df_res_bio.head(50)["n_cubes"].value_counts().to_string())
        
        print("\n Radio DBSCAN (eps) más frecuente:")
        print(df_res_bio.head(50)["eps"].value_counts().to_string())

else:
    print("\n No hay datos disponibles para generar el resumen porque la grilla no produjo resultados válidos.")


RESUMEN ESTADÍSTICO DE LOS MEJORES PARÁMETROS

 TOP 50 POR CALIDAD ESTRUCTURAL (Fórmula Matemática)
--------------------------------------------------------------
 Lentes más frecuentes:
lens
umap_2d            13
pca_2d             13
edad_madre          3
hibrido_umap_af     3
umap_15_02          3
pca_full_2d         2
eg_hijo             2
umap_5_05           2
mgAF_suple_1T       2
dif_imc             2
imc_antes           1
pca_2               1
umap_30_01          1
mgAF_total          1
pca_1               1

 Resolución (n_cubes) más frecuente:
n_cubes
20    16
12    12
16    11
8     11

 Radio DBSCAN (eps) más frecuente:
eps
1.5    45
2.0     4
1.0     1


 TOP 50 POR RELEVANCIA BIOLÓGICA (Varianza Nutricional/Clínica)
--------------------------------------------------------------
 Lentes más frecuentes:
lens
eg_hijo          32
pn_hijo           6
umap_30_01        5
densidad          3
excentricidad     2
dias_saf          2

 Resolución (n_cubes) más frecuente:
n_cubes
8